## **Query ChEMBL37 Database for Approved Small Molecule Drugs**

In [1]:
# Import modules
import adbc_driver_postgresql
import adbc_driver_postgresql.dbapi
import pandas as pd

# Expand to see all columns
pd.set_option("display.max_columns", None)

# Print versions
print(f"Pandas Version: {pd.__version__}")
print(f"ADBC Driver Version: {adbc_driver_postgresql.__version__}")

Pandas Version: 3.0.2
ADBC Driver Version: 1.11.0


### **Establish Connection to ChEMBL SQL Database**

In [2]:
uri = "postgresql://localhost/chembl37"
try:
    conn = adbc_driver_postgresql.dbapi.connect(uri)
    with conn.cursor() as cur:
        cur.execute("SELECT 1")
        assert cur.fetchone() == (1,)
    print("Database connected successfully")
except adbc_driver_postgresql.dbapi.Error as e:
    print(f"Database not connected successfully: {e}")

Database connected successfully


### **Explore Available Tables in ChEMBL**

Most tutorials online I've come across only show a specific part of the query. This makes sense as the intent of these tutorials are to teach a specific task or function. I was a little more curious about what else is available in the database. After a bit of tinkering, I found a query to identify all the table names and print them. This was helpful in the subsequent cells in determining which tables to query and merge.

In [3]:
sql_list_tables = """
SELECT table_name
FROM information_schema.tables
    WHERE table_schema = 'public'
ORDER BY table_name
"""
with conn.cursor() as cur:
    cur.execute(sql_list_tables)
    chembl_tables_df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
print(chembl_tables_df["table_name"].values)

<ArrowStringArray>
[                'action_type',                  'activities',
         'activity_properties',               'activity_smid',
        'activity_stds_lookup',               'activity_supp',
           'activity_supp_map',             'assay_class_map',
        'assay_classification',            'assay_parameters',
                  'assay_type',                      'assays',
          'atc_classification',               'binding_sites',
     'bio_component_sequences',           'bioassay_ontology',
   'biotherapeutic_components',             'biotherapeutics',
             'cell_dictionary',            'chembl_id_lookup',
              'chembl_release',             'component_class',
           'component_domains',                'component_go',
         'component_sequences',          'component_synonyms',
         'compound_properties',            'compound_records',
  'compound_structural_alerts',         'compound_structures',
     'confidence_score_lookup',     

### **Determine Available Columns in the Molecular Dictionary Table**

In [4]:
sql_table = "molecule_dictionary"
sql_list_columns = """
SELECT column_name
FROM information_schema.columns
    WHERE table_name = 'molecule_dictionary'
ORDER BY ordinal_position
"""
with conn.cursor() as cur:
    cur.execute(sql_list_columns)
    chembl_molecule_dictionary_df = pd.DataFrame(
        cur.fetchall(), columns=[desc[0] for desc in cur.description]
    )
print(f"Table name: {sql_table}")
print(f"Columns: {chembl_molecule_dictionary_df['column_name'].values}")

Table name: molecule_dictionary
Columns: <ArrowStringArray>
[            'molregno',            'pref_name',            'chembl_id',
            'max_phase',     'therapeutic_flag',     'dosed_ingredient',
       'structure_type',        'molecule_type',       'first_approval',
                 'oral',           'parenteral',              'topical',
    'black_box_warning',      'natural_product',       'first_in_class',
            'chirality',              'prodrug',       'inorganic_flag',
            'usan_year',    'availability_type',            'usan_stem',
         'polymer_flag',         'usan_substem', 'usan_stem_definition',
       'withdrawn_flag',       'chemical_probe',               'orphan',
           'veterinary']
Length: 28, dtype: str


In [5]:
sql_query = """
SELECT DISTINCT
-- Molecule Dictionary
m.molregno, m.pref_name, m.chembl_id, m.max_phase, m.therapeutic_flag, m.dosed_ingredient,
m.structure_type, m.molecule_type, m.first_approval, m.oral, m.parenteral, m.topical,
m.black_box_warning, m.natural_product, m.first_in_class, m.chirality, m.prodrug, m.inorganic_flag,
m.usan_year, m.availability_type, m.usan_stem, m.polymer_flag, m.usan_substem,
m.usan_stem_definition, m.withdrawn_flag, m.chemical_probe, m.orphan, m.veterinary
FROM molecule_dictionary m
-- WHERE m.max_phase = 4
-- AND m.molecule_type = 'Small molecule'
-- AND m.molecule_type = 'Unknown'
-- AND m.inorganic_flag != 1
"""
with conn.cursor() as cur:
    cur.execute(sql_query)
    chembl_molecule_dictionary_df = pd.DataFrame(
        cur.fetchmany(5),  # cur.fetchall(),
        columns=[desc[0] for desc in cur.description],
    )
print(chembl_molecule_dictionary_df.shape)
chembl_molecule_dictionary_df.head()
# Total molecules: 2,878,135
# Total approved: 4,005
# Total small molecules: 1,920,603 - note: Unknown identifier returns some with valid
# small molecule SMILES (Total count: 390,341) - 2,310,944 actual "small molecules"
# Total approved small molecules: 3,280 - note: 6 additional with Unknown identifier
# with a valid SMILES
# Total inorganic molecules: 204 - note: cisplatin & arsenic trioxide are only antineoplastics

(5, 28)


,molregno,pref_name,chembl_id,max_phase,therapeutic_flag,dosed_ingredient,structure_type,molecule_type,first_approval,oral,parenteral,topical,black_box_warning,natural_product,first_in_class,chirality,prodrug,inorganic_flag,usan_year,availability_type,usan_stem,polymer_flag,usan_substem,usan_stem_definition,withdrawn_flag,chemical_probe,orphan,veterinary
0,1,None,CHEMBL6329,None,0,0,MOL,Small molecule,None,0,0,0,0,0,-1,-1,-1,-1,None,-1,None,0,None,None,0,0,-1,-1
1,2,None,CHEMBL6328,None,0,0,MOL,Small molecule,None,0,0,0,0,0,-1,-1,-1,-1,None,-1,None,0,None,None,0,0,-1,-1
2,3,None,CHEMBL265667,None,0,0,MOL,Small molecule,None,0,0,0,0,0,-1,-1,-1,-1,None,-1,None,0,None,None,0,0,-1,-1
3,4,None,CHEMBL6362,None,0,0,MOL,Small molecule,None,0,0,0,0,0,-1,-1,-1,-1,None,-1,None,0,None,None,0,0,-1,-1
4,5,None,CHEMBL267864,None,0,0,MOL,Small molecule,None,0,0,0,0,0,-1,-1,-1,-1,None,-1,None,0,None,None,0,0,-1,-1


### **Determine Available Columns in the Compound Structures and Compound Properties Tables**

In [6]:
sql_table = "compound_structures"
sql_list_columns = """
SELECT column_name
FROM information_schema.columns
    WHERE table_name = 'compound_structures'
ORDER BY ordinal_position
"""
with conn.cursor() as cur:
    cur.execute(sql_list_columns)
    chembl_compound_structures_df = pd.DataFrame(
        cur.fetchall(), columns=[desc[0] for desc in cur.description]
    )
print(f"Table name: {sql_table}")
print(f"Columns: {chembl_compound_structures_df['column_name'].values}")

Table name: compound_structures
Columns: <ArrowStringArray>
[          'molregno',            'molfile',     'standard_inchi',
 'standard_inchi_key',   'canonical_smiles']
Length: 5, dtype: str


In [7]:
sql_table = "compound_properties"
sql_list_columns = """
SELECT column_name
FROM information_schema.columns
    WHERE table_name = 'compound_properties'
ORDER BY ordinal_position
"""
with conn.cursor() as cur:
    cur.execute(sql_list_columns)
    chembl_compound_properties_df = pd.DataFrame(
        cur.fetchall(), columns=[desc[0] for desc in cur.description]
    )
print(f"Table name: {sql_table}")
print(f"Columns: {chembl_compound_properties_df['column_name'].values}")

Table name: compound_properties
Columns: <ArrowStringArray>
[          'molregno',        'mw_freebase',              'alogp',
                'hba',                'hbd',                'psa',
                'rtb',           'ro3_pass', 'num_ro5_violations',
           'full_mwt',     'aromatic_rings',        'heavy_atoms',
       'qed_weighted',    'full_molformula',  'np_likeness_score']
Length: 15, dtype: str


In [8]:
sql_query = """
SELECT DISTINCT
-- Compound Structures
s.molregno, s.canonical_smiles,
-- Compound Properties
p.mw_freebase, p.alogp, p.hba, p.hbd, p.psa, p.rtb, p.ro3_pass,
p.num_ro5_violations, p.full_mwt, p.aromatic_rings, p.heavy_atoms,
p.qed_weighted, p.full_molformula, p.np_likeness_score
-- Join tables
FROM compound_structures s
    INNER JOIN compound_properties p ON s.molregno = p.molregno
"""
with conn.cursor() as cur:
    cur.execute(sql_query)
    chembl_compound_df = pd.DataFrame(
        cur.fetchmany(5),  # cur.fetchall(),
        columns=[desc[0] for desc in cur.description],
    )
print(chembl_compound_df.shape)
chembl_compound_df.head()
# Total Compounds with SMILES: 2,854,815

(5, 16)


,molregno,canonical_smiles,mw_freebase,alogp,hba,hbd,psa,rtb,ro3_pass,num_ro5_violations,full_mwt,aromatic_rings,heavy_atoms,qed_weighted,full_molformula,np_likeness_score
0,1,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccccc1Cl,341.75,2.11,5,1,84.82,3,N,0,341.75,3,24,0.74,C17H12ClN3O3,-1.56
1,2,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(C#N)cc1,332.32,1.33,6,1,108.61,3,N,0,332.32,3,25,0.73,C18H12N4O3,-1.59
2,3,Cc1cc(-n2ncc(=O)[nH]c2=O)cc(C)c1C(O)c1ccc(Cl)cc1,357.80,2.27,5,2,87.98,3,N,0,357.80,3,25,0.75,C18H16ClN3O3,-0.82
3,4,Cc1ccc(C(=O)c2ccc(-n3ncc(=O)[nH]c3=O)cc2)cc1,307.31,1.46,5,1,84.82,3,N,0,307.31,3,23,0.74,C17H13N3O3,-1.10
4,5,Cc1cc(-n2ncc(=O)[nH]c2=O)ccc1C(=O)c1ccc(Cl)cc1,341.75,2.11,5,1,84.82,3,N,0,341.75,3,24,0.74,C17H12ClN3O3,-1.49


### **Determine Available Columns in the WHO ATC Classificaton at the Molecule Level**

In [9]:
sql_table = "molecule_atc_classification"
sql_list_columns = """
SELECT column_name
FROM information_schema.columns
    WHERE table_name = 'molecule_atc_classification'
ORDER BY ordinal_position
"""
with conn.cursor() as cur:
    cur.execute(sql_list_columns)
    chembl_compound_structures_df = pd.DataFrame(
        cur.fetchall(), columns=[desc[0] for desc in cur.description]
    )
print(f"Table name: {sql_table}")
print(f"Columns: {chembl_compound_structures_df['column_name'].values}")

Table name: molecule_atc_classification
Columns: <ArrowStringArray>
['mol_atc_id', 'level5', 'molregno']
Length: 3, dtype: str


In [10]:
sql_query = """
SELECT DISTINCT
-- ATC Classification
a.molregno, a.mol_atc_id, a.level5
FROM molecule_atc_classification a
--    WHERE a.level5 LIKE 'L%'
"""
# ATC Level 1 = L -> Antineoplastic and Immunomodulating Agents
with conn.cursor() as cur:
    cur.execute(sql_query)
    chembl_atc_class_df = pd.DataFrame(
        cur.fetchall(), columns=[desc[0] for desc in cur.description]
    )
print(chembl_atc_class_df.shape)
chembl_atc_class_df.head()
# Total antineoplastic: 516

(4567, 3)


,molregno,mol_atc_id,level5
0,674645,104083,N01AB08
1,3283076,103121,S03AA06
2,28976,101305,R03AA01
3,222702,100963,G01AF13
4,675038,101172,G02CB03


### **Create a Query to Merge Structure, Compound Properties, Molecular Dictionary, and ATC Classification**

In [11]:
sql_query = """
SELECT
    m.molregno, s.canonical_smiles,
    -- Molecule Dictionary
    m.chembl_id, m.pref_name, m.max_phase::int, m.therapeutic_flag, m.dosed_ingredient,
    m.structure_type, m.molecule_type, m.first_approval::text, m.oral, m.parenteral, m.topical,
    m.black_box_warning, m.natural_product, m.first_in_class, m.chirality, m.prodrug,
    m.inorganic_flag, m.usan_year::text, m.availability_type::text, m.usan_stem, m.polymer_flag,
    m.usan_substem, m.usan_stem_definition, m.withdrawn_flag, m.chemical_probe, m.orphan,
    m.veterinary,
    -- Compound Properties
    p.mw_freebase, p.alogp, p.hba::text, p.hbd::text, p.psa, p.rtb::text, p.ro3_pass,
    p.num_ro5_violations::text, p.full_mwt, p.aromatic_rings::text, p.heavy_atoms::text,
    p.qed_weighted, p.full_molformula, p.np_likeness_score,
    -- ATC Classification
    STRING_AGG(a.mol_atc_id::text, ',' ORDER BY a.mol_atc_id) AS mol_atc_id,
    STRING_AGG(a.level5, ',' ORDER BY a.level5) AS atc_level5
FROM molecule_dictionary m
    INNER JOIN compound_structures s ON m.molregno = s.molregno
    INNER JOIN compound_properties p ON m.molregno = p.molregno
    LEFT JOIN molecule_atc_classification a ON m.molregno = a.molregno
WHERE m.max_phase = 4
GROUP BY
    m.molregno, s.canonical_smiles,
    m.chembl_id, m.pref_name, m.max_phase, m.therapeutic_flag, m.dosed_ingredient,
    m.structure_type, m.molecule_type, m.first_approval, m.oral, m.parenteral,
    m.topical, m.black_box_warning, m.natural_product, m.first_in_class,
    m.chirality, m.prodrug, m.inorganic_flag, m.usan_year, m.availability_type,
    m.usan_stem, m.polymer_flag, m.usan_substem, m.usan_stem_definition,
    m.withdrawn_flag, m.chemical_probe, m.orphan, m.veterinary,
    p.mw_freebase, p.alogp, p.hba, p.hbd, p.psa, p.rtb, p.ro3_pass,
    p.num_ro5_violations, p.full_mwt, p.aromatic_rings, p.heavy_atoms,
    p.qed_weighted, p.full_molformula, p.np_likeness_score
"""
with conn.cursor() as cur:
    cur.execute(sql_query)
    chembl_df = pd.DataFrame(cur.fetchall(), columns=[desc[0] for desc in cur.description])
print(chembl_df.shape)
chembl_df.head()
# Total molecules: 2,878,135
# Total molecules with SMILES: 2,854,815
# Total approved molecules: 3,229 (Organic: 3,176)

(3417, 45)


,molregno,canonical_smiles,chembl_id,pref_name,max_phase,therapeutic_flag,dosed_ingredient,structure_type,molecule_type,first_approval,oral,parenteral,topical,black_box_warning,natural_product,first_in_class,chirality,prodrug,inorganic_flag,usan_year,availability_type,usan_stem,polymer_flag,usan_substem,usan_stem_definition,withdrawn_flag,chemical_probe,orphan,veterinary,mw_freebase,alogp,hba,hbd,psa,rtb,ro3_pass,num_ro5_violations,full_mwt,aromatic_rings,heavy_atoms,qed_weighted,full_molformula,np_likeness_score,mol_atc_id,atc_level5
0,97,COc1cc2nc(N3CCN(C(=O)c4ccco4)CC3)nc(N)c2cc1OC,CHEMBL2,PRAZOSIN,4,1,0,MOL,Small molecule,1976,1,0,0,0,1,0,2,0,0,1968,1,-azosin,0,-azosin,antihypertensives (prazosin type),0,0,0,0,383.41,1.78,8,1,106.95,4,N,0,383.41,3,28,0.73,C19H21N5O4,-1.29,104806,C02CA01
1,115,CN1CCC[C@H]1c1cccnc1,CHEMBL3,NICOTINE,4,1,1,MOL,Small molecule,1984,1,0,1,0,1,0,1,0,0,1985,2,NaN,0,NaN,NaN,0,0,0,0,162.24,1.85,2,0,16.13,1,Y,0,162.24,1,12,0.63,C10H14N2,-0.41,100802,N07BA01
2,146,CC1COc2c(N3CCN(C)CC3)c(F)cc3c(=O)c(C(=O)O)cn1c23,CHEMBL4,OFLOXACIN,4,1,1,MOL,Small molecule,1990,1,1,1,1,0,0,0,0,0,1984,1,-oxacin,0,-oxacin,antibacterials (quinolone derivatives),0,0,0,0,361.37,1.54,6,1,75.01,2,N,0,361.37,2,26,0.87,C18H20FN3O4,-0.21,"100803,100804,104209","J01MA01,S01AE01,S02AA16"
3,147,CCn1cc(C(=O)O)c(=O)c2ccc(C)nc21,CHEMBL5,NALIDIXIC ACID,4,1,1,MOL,Small molecule,1964,1,0,0,0,1,0,2,0,0,1962,0,nal-,0,nal-,narcotic agonists/antagonists (normorphine type),0,0,0,0,232.24,1.42,4,1,72.19,2,N,0,232.24,2,17,0.85,C12H12N2O3,-0.98,103524,J01MB02
4,173,COc1ccc2c(c1)c(CC(=O)O)c(C)n2C(=O)c1ccc(Cl)cc1,CHEMBL6,INDOMETHACIN,4,1,1,MOL,Small molecule,1965,1,1,1,1,1,0,2,0,0,1963,1,NaN,0,NaN,NaN,0,0,0,0,357.79,3.93,4,1,68.53,4,N,0,357.79,3,25,0.77,C19H16ClNO4,-0.72,"100690,101780,103538,103539,104689","C01EB03,M01AB01,M01AB51,M02AA23,S01BC01"
